# 01. 기초: pass@k와 reasoning effectiveness

목표: Thinking to Recall 논문에서 사용하는 pass@k와 Omega 지표를 작은 예제로 계산합니다.

실행 방법: 위에서부터 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. 샘플 답변과 정답 준비

pass@k는 k개 샘플 중 하나라도 정답이 있는지 봅니다. 아래 toy data는 reasoning OFF와 ON에서 각각 10개 답변을 샘플링했다고 가정합니다.

In [ ]:
questions = [
    {
        "id": "q1",
        "answer": "1940",
        "off": ["1938", "1939", "1941", "1936", "unknown", "1942", "1937", "1943", "1935", "1944"],
        "on": ["1938", "1940", "1939", "1941", "1940", "unknown", "1942", "1940", "1937", "1943"],
    },
    {
        "id": "q2",
        "answer": "Birendra Bir Bikram Shah Dev",
        "off": ["Jitari Malla", "Mahendra", "Prithvi Narayan", "Tribhuvan", "Gyanendra", "Mahendra", "unknown", "Rajendra", "Surendra", "Pratap"],
        "on": ["Jitari Malla", "Mahendra", "Birendra Bir Bikram Shah Dev", "Tribhuvan", "Birendra Bir Bikram Shah Dev", "Gyanendra", "Mahendra", "Birendra Bir Bikram Shah Dev", "unknown", "Rajendra"],
    },
    {
        "id": "q3",
        "answer": "Ada Lovelace",
        "off": ["Ada Lovelace", "Charles Babbage", "Grace Hopper", "Alan Turing", "Ada Lovelace", "unknown", "Lovelace", "Babbage", "Hopper", "Byron"],
        "on": ["Ada Lovelace", "Ada Lovelace", "Charles Babbage", "Ada Lovelace", "Grace Hopper", "Ada Lovelace", "unknown", "Lovelace", "Babbage", "Hopper"],
    },
]


def normalize_answer(text):
    return " ".join(text.lower().split())


def is_correct(sample, answer):
    return normalize_answer(sample) == normalize_answer(answer)


for item in questions:
    print(item["id"], "answer=", item["answer"])


## 2. pass@k 계산

아래 구현은 각 질문에서 앞쪽 k개 샘플 중 정답이 하나라도 있으면 성공으로 둡니다. 실제 논문에서는 더 많은 샘플과 통계 추정을 사용합니다.

In [ ]:
def pass_at_k(dataset, mode, k):
    successes = []
    for item in dataset:
        sampled = item[mode][:k]
        success = any(is_correct(sample, item["answer"]) for sample in sampled)
        successes.append(success)
    return sum(successes) / len(successes)


print("k | pass@k OFF | pass@k ON")
print("--- | --- | ---")
for k in [1, 2, 3, 5, 10]:
    print(f"{k:2d} | {pass_at_k(questions, 'off', k):.3f} | {pass_at_k(questions, 'on', k):.3f}")

## 3. Omega 지표 계산

논문은 큰 k에서의 개선을 더 중요하게 보기 위해 k로 가중한 상대 개선 평균을 사용합니다. 아래 함수는 README에 적은 형태의 toy Omega를 계산합니다.

In [ ]:
def reasoning_effectiveness_omega(dataset, max_k):
    numerator = 0.0
    denominator = sum(range(1, max_k + 1))
    for k in range(1, max_k + 1):
        off = pass_at_k(dataset, "off", k)
        on = pass_at_k(dataset, "on", k)
        if off == 0:
            # off가 0이면 상대 개선이 무한대가 되므로 학습 예제에서는 건너뜁니다.
            continue
        numerator += k * ((on - off) / off)
    return numerator / denominator


for max_k in [3, 5, 10]:
    print(f"Omega(N={max_k}) = {reasoning_effectiveness_omega(questions, max_k):.3f}")

## 4. 해석

reasoning ON은 pass@1뿐 아니라 큰 k에서도 더 많은 정답 경로를 만듭니다. 이것이 논문이 말하는 capability boundary 확장의 직관입니다. 모델이 사실을 전혀 모르는 것이 아니라, reasoning 없이 꺼내기 어려운 낮은 확률 영역에 정답이 숨어 있을 수 있습니다.